# ReceiptRadar - Pipeline Exploration & Modeling Walkthrough
This notebook demonstrates the core stages of ReceiptRadar: OCR, LLM extraction, ML category classification, and anomaly detection.

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

from src.ocr.engine import OCREngine
from src.parsing.parser import LLMParser
from src.models.classifier import CategoryClassifier
from src.models.anomaly import ExpenseAnomalyDetector
from src.models.forecaster import SpendingForecaster
from src.models.recommender import SavingsRecommender
from src.api.config import SAMPLE_RECEIPTS_DIR
from PIL import Image

## 1. Document OCR & LLM Parsing

In [ ]:
ocr = OCREngine()
llm = LLMParser()
receipt_file = list(SAMPLE_RECEIPTS_DIR.glob('*.png'))[0]
print('Processing:', receipt_file.name)

ocr_out = ocr.extract_text(receipt_file)
parsed = llm.parse(ocr_out['raw_text'])
print('Merchant:', parsed.merchant_name)
print('Total:', parsed.total_amount)
print('Items:', len(parsed.items))

## 2. ML Classification & Anomaly Audit

In [ ]:
clf = CategoryClassifier()
ano = ExpenseAnomalyDetector()

items_desc = ' '.join([i.description for i in parsed.items])
pred_cat, conf, _ = clf.predict(parsed.merchant_name, items_desc)
risk = ano.audit_transaction(
    amount=parsed.total_amount,
    category=pred_cat,
    merchant=parsed.merchant_name,
    txn_date_str=parsed.date,
)
print(f'Category: {pred_cat} ({conf*100:.1f}%)')
print(f'Audit Risk: {risk.risk_severity} (Score: {risk.anomaly_score*100:.1f}%)')